In [ ]:
!pip install langchain langchain-openai langchain-community langchain-text-splitters faiss-cpu numpy python-dotenv rank-bm25 pymupdf

In [23]:
path = r"C:\AYUSH\Rag\Rag-Lanchain\book\java21.pdf"

In [36]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_classic.retrievers.ensemble import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.embeddings import OllamaEmbeddings
from dotenv import load_dotenv

In [28]:
from langchain_classic.chains import RetrievalQA

In [49]:
load_dotenv()
import os
api_key = os.getenv("GOOGLE_API_KEY")

In [16]:
loader = PyPDFLoader(path)
documents = loader.load()

In [18]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 512,
    chunk_overlap=50
)
splits = text_splitter.split_documents(documents)

In [20]:
!ollama list

NAME                        ID              SIZE      MODIFIED     
phi3:mini                   4f2222927938    2.2 GB    6 weeks ago     
gemma3:270m                 e7d36fb2c3b3    291 MB    2 months ago    
mxbai-embed-large:latest    468836162de7    669 MB    2 months ago    
tinyllama:latest            2644915ede35    637 MB    2 months ago    
nomic-embed-text:latest     0a109f422b47    274 MB    2 months ago    


In [24]:
embedding = OllamaEmbeddings(
    model = "nomic-embed-text:latest"
)
#Dense retriever (semantic)
vector_store = Chroma.from_documents(splits, embedding)
dense_retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# Sparse retriever (keyword - BM25)
sparse_retriever = BM25Retriever.from_documents(splits)
sparse_retriever.k = 3

In [41]:
alpha : float = 0.5
# Hybrid retriever
hybrid_retriever = EnsembleRetriever(
    retrievers = [dense_retriever, sparse_retriever],
    weights=[alpha, 1-alpha]
)

In [42]:
llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash",
    api_key = api_key
)

In [32]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=hybrid_retriever,
    return_source_documents=True
)

In [ ]:
query = "What is the main topic of this document?"
result = qa_chain.invoke({"query": query})

print("Answer:", result["result"])
print("\nSources:")
for doc in result["source_documents"]:
    print(f"- {doc.page_content[:200]}...")